In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
chest_xray_pneumonia_path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print('Path of dataset = ' + chest_xray_pneumonia_path)
print('Data source import complete.')


# 🫁 Chest X-Ray Pneumonia Detection
### Binary Classification: NORMAL vs PNEUMONIA

**Approach:**
1. Custom CNN from scratch (to understand the fundamentals)
2. Transfer Learning with EfficientNetB0 (production-grade performance)

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia)  
**Total Images:** ~5,856 | **Classes:** NORMAL, PNEUMONIA

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

## 1. 📦 Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import EfficientNetB0

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Config
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS_CNN = 30
EPOCHS_TL  = 20
LR         = 1e-4

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. 📂 Data Paths

In [ ]:
BASE_DIR  = '//kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR   = os.path.join(BASE_DIR, 'val')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

# Verify paths exist
for name, path in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    print(f'{name}: {path} — exists: {os.path.exists(path)}')

## 3. EDA — Class Distribution & Sample Images

In [ ]:
def count_images(directory):
    """Count images per class in a directory, always sorted NORMAL then PNEUMONIA."""
    counts = {}
    for cls in sorted(os.listdir(directory)):  # sorted = NORMAL, PNEUMONIA
        cls_path = os.path.join(directory, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len(os.listdir(cls_path))
    return counts
train_counts = count_images(TRAIN_DIR)
val_counts   = count_images(VAL_DIR)
test_counts  = count_images(TEST_DIR)
print('Train:', train_counts)
print('Val:  ', val_counts)
print('Test: ', test_counts)



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLOR_MAP = {
    'NORMAL':    '#8fbf9f',   # soft muted green
    'PNEUMONIA': '#c88a8a'    # soft muted rose
}

splits = [
    ('Train',      train_counts),
    ('Validation', val_counts),
    ('Test',       test_counts),
]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'<b>{s}</b> Set' for s, _ in splits],
    horizontal_spacing=0.08,
)

shown_in_legend = set()

for col, (split_name, counts) in enumerate(splits, start=1):
    for cls in ['NORMAL', 'PNEUMONIA']:
        val = counts[cls]
        show_legend = cls not in shown_in_legend
        shown_in_legend.add(cls)

        fig.add_trace(
            go.Bar(
                name=cls,
                x=[cls],
                y=[val],
                marker=dict(
                    color=COLOR_MAP[cls],
                    line=dict(color='rgba(0,0,0,0.25)', width=1),
                    opacity=0.9,
                ),
                text=[f'<b>{val:,}</b>'],
                textposition='outside',
                textfont=dict(size=13, color='#3d3d3d'),
                width=0.45,
                showlegend=show_legend,
                legendgroup=cls,
                hovertemplate=(
                    f'<b>{split_name} — {cls}</b><br>'
                    f'Count: <b>{val:,}</b><br>'
                    f'Share: <b>{val/sum(counts.values())*100:.1f}%</b>'
                    '<extra></extra>'
                ),
            ),
            row=1, col=col,
        )

fig.update_layout(
    title=dict(
        text='Class Distribution Across Splits',
        font=dict(size=20, color='#2c3e50', family='Georgia, serif'),
        x=0.5, xanchor='center', y=0.97,
    ),
    legend=dict(
        title=dict(text='Class', font=dict(size=12)),
        font=dict(size=12),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.1)',
        borderwidth=1,
        x=1.01, y=1,
    ),
    paper_bgcolor='#fafaf8',
    plot_bgcolor='#fafaf8',
    barmode='group',
    bargap=0.25,
    font=dict(family='Georgia, serif', color='#3d3d3d'),
    height=480,
    margin=dict(t=80, b=60, l=60, r=120),
    hoverlabel=dict(
        bgcolor='white',
        font_size=13,
        bordercolor='rgba(0,0,0,0.15)',
    ),
)

# Clean axes: remove gridlines on x, soft gridlines on y
for col in range(1, 4):
    fig.update_xaxes(
        showgrid=False,
        tickfont=dict(size=12),
        row=1, col=col,
    )
    fig.update_yaxes(
        gridcolor='rgba(0,0,0,0.07)',
        zeroline=False,
        title_text='Number of Images' if col == 1 else '',
        tickfont=dict(size=11),
        row=1, col=col,
    )

fig.show()

# Class imbalance ratio (unchanged)
ratio = train_counts['PNEUMONIA'] / train_counts['NORMAL']
print(f'\n⚠️  Class imbalance ratio (PNEUMONIA/NORMAL): {ratio:.2f}x')
print(f'   NORMAL: {train_counts["NORMAL"]} | PNEUMONIA: {train_counts["PNEUMONIA"]}')

**→ PNEUMONIA is the majority class — using class_weight to balance training.**

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.patches as mpatches

def show_sample_images(directory, n=5):
    """Display sample images from each class with enhanced styling."""
    classes = ['NORMAL', 'PNEUMONIA']

    CLASS_STYLE = {
        'NORMAL':    {'color': '#8fbf9f', 'label': '✦  NORMAL'},
        'PNEUMONIA': {'color': '#c88a8a', 'label': '✦  PNEUMONIA'},
    }

    fig, axes = plt.subplots(
        2, n,
        figsize=(n * 3.2, 8),
        facecolor='#1a1a1a',
        gridspec_kw={'hspace': 0.08, 'wspace': 0.05}
    )

    for row, cls in enumerate(classes):
        cls_dir = os.path.join(directory, cls)
        images  = np.random.choice(os.listdir(cls_dir), n, replace=False)
        style   = CLASS_STYLE[cls]

        for col, img_name in enumerate(images):
            ax = axes[row, col]
            img = load_img(os.path.join(cls_dir, img_name), target_size=(224, 224))
            ax.imshow(img, cmap='gray', aspect='auto')
            ax.set_xticks([])
            ax.set_yticks([])

            # Colored border per class
            for spine in ax.spines.values():
                spine.set_edgecolor(style['color'])
                spine.set_linewidth(2.2)

            # Class label badge on the leftmost image only
            if col == 0:
                ax.text(
                    -0.06, 0.5, style['label'],
                    transform=ax.transAxes,
                    fontsize=12, fontweight='bold',
                    color=style['color'],
                    va='center', ha='right',
                    rotation=90,
                    fontfamily='monospace',
                )

            # Subtle image index
            ax.text(
                0.97, 0.03, f'{col + 1}',
                transform=ax.transAxes,
                fontsize=8, color='rgba(255,255,255,0.35)' if False else '#666666',
                ha='right', va='bottom',
                fontfamily='monospace',
            )

    # Title
    fig.suptitle(
        'Sample X-Ray Images',
        fontsize=17,
        fontweight='bold',
        color='#e8e8e8',
        fontfamily='DejaVu Serif',
        y=1.01,
    )

    # Legend
    legend_patches = [
        mpatches.Patch(facecolor=CLASS_STYLE[c]['color'], label=c, linewidth=0)
        for c in classes
    ]

    plt.tight_layout()
    plt.show()

show_sample_images(TRAIN_DIR)

## 4. Data Generators & Augmentation

In [ ]:
# --- Augmentation only on training data ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# --- Generators ---
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f'Class indices: {train_gen.class_indices}')
# NORMAL=0, PNEUMONIA=1

In [ ]:
# --- Compute class weights to handle imbalance ---
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weight_dict = dict(enumerate(class_weights))
print(f'Class weights: {class_weight_dict}')
# Higher weight on NORMAL since it has fewer samples

## 5. **Model 1 — Custom CNN from Scratch**

Architecture: **3 Conv blocks → Flatten → Dense → Output**  
Each block: Conv2D → BatchNorm → MaxPool → Dropout

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3)):
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Classifier head — GAP avoids param explosion from Flatten
        # Flatten() → 28*28*128 = 100,352 → Dense(256) = 25.6M params!
        # GlobalAveragePooling2D → 128 → Dense(256) = only 33K params
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(1, activation='sigmoid')  # Binary output
    ], name='CustomCNN')
    return model

cnn_model = build_custom_cnn()
cnn_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
cnn_model.summary()

In [ ]:
# --- Callbacks ---
callbacks_cnn = [
    EarlyStopping(monitor='val_auc', patience=7, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_custom_cnn.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1)
]

# --- Train ---
print('Training Custom CNN...')
history_cnn = cnn_model.fit(
    train_gen,
    epochs=EPOCHS_CNN,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=callbacks_cnn,
    verbose=1
)

## 6. 📈 Training Curves — Custom CNN

In [ ]:
def plot_history(history, model_name='Model'):
    """Plot training & validation accuracy, loss, and AUC."""
    metrics = ['accuracy', 'loss', 'auc']
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, metric in zip(axes, metrics):
        ax.plot(history.history[metric],        label=f'Train {metric}', linewidth=2)
        ax.plot(history.history[f'val_{metric}'], label=f'Val {metric}', linewidth=2, linestyle='--')
        ax.set_title(f'{metric.capitalize()}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.suptitle(f'{model_name} — Training History', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_history(history_cnn, 'Custom CNN')

## 7. 📊 Evaluation — Custom CNN

In [ ]:
def evaluate_model(model, test_gen, model_name='Model'):
    """Full evaluation: loss/acc, confusion matrix, classification report, ROC-AUC."""

    # Reset generator
    test_gen.reset()

    # Predictions
    y_pred_proba = model.predict(test_gen, verbose=1).flatten()
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    y_true       = test_gen.classes
    class_names  = list(test_gen.class_indices.keys())  # ['NORMAL', 'PNEUMONIA']

    # --- Metrics ---
    test_loss, test_acc, test_auc = model.evaluate(test_gen, verbose=0)
    roc_auc = roc_auc_score(y_true, y_pred_proba)

    print(f'\n{"="*50}')
    print(f'  {model_name} — Test Results')
    print(f'{"="*50}')
    print(f'  Accuracy : {test_acc:.4f}')
    print(f'  AUC      : {roc_auc:.4f}')
    print(f'  Loss     : {test_loss:.4f}')
    print(f'{"="*50}\n')

    # --- Classification Report ---
    print(classification_report(y_true, y_pred, target_names=class_names))

    # --- Confusion Matrix ---
    cm = confusion_matrix(y_true, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Confusion matrix heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], linewidths=0.5, linecolor='gray')
    axes[0].set_title(f'{model_name} — Confusion Matrix', fontweight='bold')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    axes[1].plot(fpr, tpr, lw=2, label=f'ROC AUC = {roc_auc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    axes[1].set_title(f'{model_name} — ROC Curve', fontweight='bold')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].legend(loc='lower right')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return {'accuracy': test_acc, 'auc': roc_auc, 'loss': test_loss}

cnn_results = evaluate_model(cnn_model, test_gen, 'Custom CNN')

## 8. ⚡ Model 2 — Transfer Learning with EfficientNetB0

**Strategy:**
- Phase 1: Freeze EfficientNetB0 base → train only classifier head (5 epochs, fast)
- Phase 2: Unfreeze top layers → fine-tune end-to-end (lower LR)

In [ ]:
def build_efficientnet_model(input_shape=(224, 224, 3), freeze_base=True):
    # Load pretrained base (ImageNet weights)
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = not freeze_base

    # Custom head
    inputs = base_model.input
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='EfficientNetB0_Transfer')
    return model, base_model

# Note: EfficientNet has its own built-in normalization,
# so we need generators WITHOUT rescale=1./255
train_datagen_tl = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_test_datagen_tl = ImageDataGenerator()

train_gen_tl = train_datagen_tl.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=True
)
val_gen_tl = val_test_datagen_tl.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)
test_gen_tl = val_test_datagen_tl.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', shuffle=False
)

In [ ]:
# ---- Phase 1: Train head only ----
tl_model, base_model = build_efficientnet_model(freeze_base=True)
tl_model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print(f'Trainable params (head only): {tl_model.count_params():,}')

callbacks_phase1 = [
    EarlyStopping(monitor='val_auc', patience=5, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

print('Phase 1: Training classifier head...')
history_tl_p1 = tl_model.fit(
    train_gen_tl,
    epochs=10,
    validation_data=val_gen_tl,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase1,
    verbose=1
)

In [ ]:
# ---- Phase 2: Unfreeze top layers & fine-tune ----
# Unfreeze last 30 layers of the base model
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with very low LR
tl_model.compile(
    optimizer=Adam(learning_rate=LR / 10),  # 1e-5
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print(f'Trainable params (fine-tune): {sum([tf.size(w).numpy() for w in tl_model.trainable_weights]):,}')

callbacks_phase2 = [
    EarlyStopping(monitor='val_auc', patience=7, restore_best_weights=True, mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint('best_efficientnet.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1)
]

print('Phase 2: Fine-tuning top layers...')
history_tl_p2 = tl_model.fit(
    train_gen_tl,
    epochs=EPOCHS_TL,
    validation_data=val_gen_tl,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2,
    verbose=1
)

## 9. 📈 Training Curves — EfficientNetB0

In [ ]:
# Combine both phases for plotting
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

combined_history = combine_histories(history_tl_p1, history_tl_p2)

class FakeHistory:
    def __init__(self, history):
        self.history = history

plot_history(FakeHistory(combined_history), 'EfficientNetB0 (Phase 1 + 2)')

# Mark phase boundary
phase1_end = len(history_tl_p1.history['accuracy'])
print(f'Phase 1 ended at epoch {phase1_end}')

## 10. 📊 Evaluation — EfficientNetB0

In [ ]:
tl_results = evaluate_model(tl_model, test_gen_tl, 'EfficientNetB0')

## 11. 🏆 Model Comparison

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

comparison_df = pd.DataFrame({
    'Model':     ['Custom CNN', 'EfficientNetB0 (TL)'],
    'Accuracy':  [cnn_results['accuracy'], tl_results['accuracy']],
    'ROC-AUC':   [cnn_results['auc'],      tl_results['auc']],
    'Test Loss': [cnn_results['loss'],      tl_results['loss']]
})
comparison_df = comparison_df.set_index('Model')
print(comparison_df.round(4))

# ── Plotly interactive bar chart ──────────────────────────────────────────────
MODEL_COLORS = {
    'Custom CNN':          '#8fbf9f',   # same soft green from your palette
    'EfficientNetB0 (TL)': '#c88a8a',   # same soft rose
}
metrics  = ['Accuracy', 'ROC-AUC']
models   = comparison_df.index.tolist()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f'<b>{m}</b>' for m in metrics],
    horizontal_spacing=0.12,
)

shown_in_legend = set()

for col, metric in enumerate(metrics, start=1):
    for model in models:
        val         = comparison_df.loc[model, metric]
        show_legend = model not in shown_in_legend
        shown_in_legend.add(model)

        fig.add_trace(
            go.Bar(
                name=model,
                x=[model],
                y=[val],
                marker=dict(
                    color=MODEL_COLORS[model],
                    line=dict(color='rgba(0,0,0,0.25)', width=1),
                    opacity=0.9,
                ),
                text=[f'<b>{val:.4f}</b>'],
                textposition='outside',
                textfont=dict(size=13, color='#3d3d3d'),
                width=0.4,
                showlegend=show_legend,
                legendgroup=model,
                hovertemplate=(
                    f'<b>{model}</b><br>'
                    f'{metric}: <b>{{y:.4f}}</b>'
                    '<extra></extra>'
                ),
            ),
            row=1, col=col,
        )

fig.update_layout(
    title=dict(
        text='Model Comparison: Custom CNN vs Transfer Learning',
        font=dict(size=18, color='#2c3e50', family='DejaVu Serif'),
        x=0.5, xanchor='center', y=0.97,
    ),
    legend=dict(
        title=dict(text='Model', font=dict(size=12)),
        font=dict(size=12),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='rgba(0,0,0,0.1)',
        borderwidth=1,
        x=1.01, y=1,
    ),
    paper_bgcolor='#fafaf8',
    plot_bgcolor='#fafaf8',
    barmode='group',
    font=dict(family='DejaVu Serif', color='#3d3d3d'),
    height=440,
    margin=dict(t=80, b=60, l=60, r=140),
    hoverlabel=dict(
        bgcolor='white',
        font_size=13,
        bordercolor='rgba(0,0,0,0.15)',
    ),
)

for col in range(1, 3):
    fig.update_xaxes(showgrid=False, tickfont=dict(size=12), row=1, col=col)
    fig.update_yaxes(
        range=[0, 1.1],
        gridcolor='rgba(0,0,0,0.07)',
        zeroline=False,
        title_text='Score' if col == 1 else '',
        tickfont=dict(size=11),
        row=1, col=col,
    )

fig.show()

## 12. 🔬 Error Analysis — False Positives & False Negatives
Understanding where the model fails is critical for medical AI.

In [ ]:
def show_misclassified(model, generator, n=8, model_name='Model'):
    """Show misclassified images: FP and FN."""
    generator.reset()
    y_pred_proba = model.predict(generator, verbose=0).flatten()
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    y_true       = generator.classes
    filenames    = generator.filenames

    fp_idx = np.where((y_pred == 1) & (y_true == 0))[0]
    fn_idx = np.where((y_pred == 0) & (y_true == 1))[0]

    print(f'{model_name} — False Positives: {len(fp_idx)} | False Negatives: {len(fn_idx)}')
    print(f'⚠️  False Negatives are more dangerous in medical context (missed diagnosis)!')

    n_cols = min(4, max(len(fp_idx), len(fn_idx)))
    if n_cols == 0:
        print('No misclassifications found!')
        return

    ROW_STYLE = {
        'FP': {
            'idx_set': fp_idx,
            'color':   '#c9a84c',          # soft amber
            'label':   '✦  FALSE POSITIVE\nNormal → Pneumonia',
        },
        'FN': {
            'idx_set': fn_idx,
            'color':   '#c88a8a',          # soft rose (matches your palette)
            'label':   '✦  FALSE NEGATIVE\nPneumonia → Normal',
        },
    }

    fig, axes = plt.subplots(
        2, n_cols,
        figsize=(n_cols * 3.2, 8),
        facecolor='#1a1a1a',
        gridspec_kw={'hspace': 0.08, 'wspace': 0.05},
    )
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    for row, (_, style) in enumerate(ROW_STYLE.items()):
        idx_set = style['idx_set']
        color   = style['color']

        for col in range(n_cols):
            ax = axes[row, col]

            if col < len(idx_set):
                idx      = idx_set[col]
                img_path = os.path.join(TEST_DIR, filenames[idx])
                img      = load_img(img_path, target_size=(224, 224))
                ax.imshow(img, cmap='gray', aspect='auto')

                # Confidence badge at bottom
                conf = y_pred_proba[idx]
                ax.text(
                    0.5, 0.04,
                    f'conf: {conf:.2f}',
                    transform=ax.transAxes,
                    fontsize=9, color='white',
                    ha='center', va='bottom',
                    fontfamily='monospace',
                    bbox=dict(
                        boxstyle='round,pad=0.3',
                        facecolor=(0, 0, 0, 0.55),   # ← tuple (R, G, B, A) instead of 'rgba(...)'
                        edgecolor='none',
                    ),
                )

                # Colored border per error type
                for spine in ax.spines.values():
                    spine.set_edgecolor(color)
                    spine.set_linewidth(2.2)
            else:
                ax.set_facecolor('#1a1a1a')
                for spine in ax.spines.values():
                    spine.set_edgecolor('#2a2a2a')
                    spine.set_linewidth(1)

            ax.set_xticks([])
            ax.set_yticks([])

            # Row label on leftmost column
            if col == 0:
                ax.text(
                    -0.08, 0.5,
                    style['label'],
                    transform=ax.transAxes,
                    fontsize=9.5, fontweight='bold',
                    color=color,
                    va='center', ha='right',
                    rotation=90,
                    fontfamily='monospace',
                    linespacing=1.6,
                )

    fig.suptitle(
        f'{model_name}  —  Misclassified Images',
        fontsize=16, fontweight='bold',
        color='#e8e8e8',
        fontfamily='DejaVu Serif',
        y=1.01,
    )

    # Legend
    import matplotlib.patches as mpatches
    legend_patches = [
        mpatches.Patch(facecolor=s['color'], label=k.replace('_', ' '))
        for k, s in [('False Positive', ROW_STYLE['FP']), ('False Negative', ROW_STYLE['FN'])]
    ]

    plt.tight_layout()
    plt.show()

show_misclassified(tl_model, test_gen_tl, model_name='EfficientNetB0')

**The misclassified images tell an important story:**

- **Top row (FP, conf ~0.60-0.94)** — Healthy lungs the model thought were pneumonia. Notice conf: 0.94 on the last one — model was very confident but wrong. These look slightly hazy which confused it.
- **Bottom row (FN, conf ~0.04-0.15)** — Actual pneumonia the model called normal. Low confidence scores (0.04, 0.09) means the model was uncertain — these are **borderline cases even for radiologists**.


## 13. 💾 Save Best Model

In [ ]:
# Save the better performing model
tl_model.save('pneumonia_efficientnet_final.keras')
print('EfficientNetB0 model saved.')

cnn_model.save('pneumonia_custom_cnn_final.keras')
print('Custom CNN model saved.')

## 14. 📝 Conclusions

### Final Results

| Metric | Custom CNN | EfficientNetB0 (TL) | Improvement |
|--------|-----------|---------------------|-------------|
| Accuracy | 0.7179 | **0.8750** | +15.7% |
| ROC-AUC | 0.9358 | **0.9427** | +0.7% |
| Test Loss | 0.7868 | **0.2999** | -61.9% |
| NORMAL Recall | 0.25 | **0.83** | +232% |
| PNEUMONIA Recall | 1.00 | **0.90** | balanced |
| False Positives | 176 | **40** | -77% |
| False Negatives | 0 | **38** | trade-off |

---

### Key Findings

**1. Class imbalance shaped model behavior significantly**  
The dataset has a 2.89x imbalance (PNEUMONIA >> NORMAL). Without `class_weight`, the Custom CNN learned a dangerous shortcut — predicting PNEUMONIA for almost everything, achieving 100% PNEUMONIA recall but only 25% NORMAL recall. This highlights why accuracy alone is a misleading metric for imbalanced medical datasets. AUC and per-class recall are far more informative.

**2. Custom CNN vs Transfer Learning — more than just accuracy**  
Both models achieved similar AUC (~0.93-0.94), but their behavior was fundamentally different. The Custom CNN had zero False Negatives (never missed pneumonia) but 176 False Positives (over-diagnosed healthy patients). EfficientNetB0 learned a genuinely balanced representation — 40 FP and 38 FN — which is far more clinically useful as a screening tool.

**3. GlobalAveragePooling2D over Flatten is non-negotiable**  
Using `Flatten()` after 3 pooling layers with 224×224 input created a 100,352-dimensional vector → 25.6M parameters in a single Dense layer (98.8% of total model). Replacing with `GlobalAveragePooling2D` reduced this to 33K parameters, trained faster, generalized better, and is the standard approach in all modern architectures (ResNet, EfficientNet, MobileNet).

**4. The 16-image validation set is a known dataset issue**  
val_accuracy was flat at 0.50 throughout training and val_auc fluctuated wildly — both are artifacts of having only 8 images per class in the val split. EarlyStopping still worked correctly by tracking the best checkpoint, and the 624-image test set provided reliable final evaluation.

**5. Misclassified images reveal clinically hard cases**  
The False Negatives (pneumonia missed by EfficientNetB0) had low confidence scores (0.04–0.15), indicating the model was uncertain — these are genuinely ambiguous X-rays that would challenge radiologists too. The False Positives showed slight haze patterns that visually resemble early-stage pneumonia.

---

### What I Would Do Next

**Threshold tuning** — The default decision threshold of 0.5 is rarely optimal. Lowering to ~0.3 would reduce False Negatives (missed pneumonia) at the cost of more False Positives, which is the right trade-off for a medical screening tool where missing a case is more dangerous than a false alarm.

**Grad-CAM visualization** — Overlay activation heatmaps on X-rays to show *which region* of the lung the model focuses on. This is critical for clinical trust and interpretability — a model that gets the right answer for the wrong reason is not deployable.

**Stronger backbone** — EfficientNetB3 or EfficientNetV2S would likely push AUC above 0.96 with minimal code change.

**Test-time augmentation (TTA)** — Average predictions across multiple augmented versions of each test image to squeeze out extra performance without retraining.